# TP03 — Arboles de decisión - Ejercicio 3
   Bases de Datos Masivas (11088) · Gustavo An Contardi - 182818

   

## a. Tema del dataset

El dataset trata sobre la informalidad laboral de los asalariados en la Argentina. Sale
de la Encuesta Permanente de Hogares (EPH) del INDEC, primer trimestre de 2026, y cubre
los 32 aglomerados urbanos de la encuesta. Cada fila es una persona que trabaja en
relación de dependencia (obreros, empleados y servicio doméstico).

Tiene datos de la persona (región, sexo, edad, nivel educativo) y de su trabajo
(sector, rama, calificación, tamaño del establecimiento, lugar de trabajo, antigüedad,
si el empleo es temporario, horas semanales e ingreso). La clase es `informal`: vale
"si" cuando el asalariado no tiene descuento jubilatorio por ese trabajo, que es el
criterio que usa el INDEC para medir la informalidad de los asalariados.

Las filas no están expandidas por el ponderador de la encuesta. Por eso las proporciones
del dataset no son estimaciones de la población, sino de la muestra.

## b. Variable objetivo y nulos

La variable a modelar es `informal`. De 13.938 asalariados, 8.694 son formales (62,4 %)
y 5.244 informales (37,6 %). Está desbalanceada, pero de forma moderada: no hace falta
reponderar, aunque la accuracy sola no alcanza para evaluar. Un modelo que diga
"formal" a todos ya tiene 62,4 % de accuracy. Por eso uso el F1 de la clase informal
como métrica principal.

Lo más importante de este dataset son los nulos. Solo 7.971 filas (57,2 %) están
completas, y los nulos no son aleatorios: al servicio doméstico el cuestionario no le
hace las preguntas del bloque de ocupación principal. En las filas completas la
informalidad es 27,2 %, y en las que tienen algún nulo sube a 51,6 %. El servicio
doméstico tiene 79,3 % de informalidad, y de sus 1.329 filas solo 8 sobreviven a un
`dropna()`.

Si descartara las filas incompletas borraría al grupo más informal y estaría
modelando otra población. Decidí no descartar ni imputar. En las categóricas sin orden
(región, sector, rama, lugar de trabajo) el nulo pasa a ser una categoría "Sin dato".
En las numéricas y las ordinales dejo los NaN, porque el árbol de scikit-learn 1.9
los maneja solo: en cada corte elige para qué lado mandarlos.

Codifiqué en ordinal las variables que tienen un orden real (nivel educativo,
antigüedad, calificación), en one-hot las que no lo tienen y en 0/1 las binarias.
El split es estratificado, así que train y test quedan con 62,4/37,6.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)

# Misma semilla en split, folds y árboles: las configuraciones se comparan
# en igualdad de condiciones
SEMILLA = 42

df = pd.read_csv("data/asalariados_eph.csv")
print(df.shape)
nulos = df.isna().sum()
nulos = pd.DataFrame({"nulos": nulos, "%": (nulos / len(df) * 100).round(1)})
nulos[nulos["nulos"] > 0].sort_values("nulos", ascending=False)

(13938, 14)


,nulos,%
tamanio_establecimiento,3869,27.8
ingreso_mensual,2598,18.6
empleo_temporario,2314,16.6
antiguedad,1401,10.1
lugar_trabajo,1314,9.4
horas_semanales,20,0.1
calificacion,16,0.1


In [2]:
# Conteo y proporción: sirve para decir si está balanceada
print(df["informal"].value_counts())
df["informal"].value_counts(normalize=True).round(3)

informal
no    8694
si    5244
Name: count, dtype: int64


informal
no    0.624
si    0.376
Name: proportion, dtype: float64

In [3]:
# Comparo la informalidad en todo el dataset, en las filas completas, en las
# filas con algún nulo y en el servicio doméstico. Si cambia mucho, los nulos
# no son aleatorios y descartarlos cambia la población.
completa = df.notna().all(axis=1)
es_domestico = df["rama"] == "Servicio doméstico"
pct_informal = lambda s: round((s == "si").mean() * 100, 1)

print(pd.DataFrame({
    "filas": [len(df), completa.sum(), (~completa).sum(), es_domestico.sum()],
    "% informal": [pct_informal(df["informal"]),
                   pct_informal(df.loc[completa, "informal"]),
                   pct_informal(df.loc[~completa, "informal"]),
                   pct_informal(df.loc[es_domestico, "informal"])],
}, index=["todas", "completas", "con algún nulo", "servicio doméstico"]))
print("Servicio doméstico que sobrevive a un dropna():", (completa & es_domestico).sum())

                    filas  % informal
todas               13938        37.6
completas            7971        27.2
con algún nulo       5967        51.6
servicio doméstico   1329        79.3
Servicio doméstico que sobrevive a un dropna(): 8


## c. Árbol con criterio entropía y hold-out 80/20

In [4]:
# Clase positiva = informal ("si" = 1): precision, recall y F1 se calculan sobre ella
y = (df["informal"] == "si").astype(int)
X = df.drop(columns="informal").copy()

# Ordinales: estas tres sí tienen un orden real, así que las paso a 0, 1, 2...
# Así el árbol puede cortar en "nivel <= secundario completo" con un solo split.
ORDENES = {
    "nivel_educativo": ["Sin instrucción", "Primario incompleto", "Primario completo",
                        "Secundario incompleto", "Secundario completo",
                        "Superior incompleto", "Superior completo"],
    "antiguedad": ["Menos de 1 mes", "1 a 3 meses", "3 a 6 meses", "6 a 12 meses",
                   "1 a 5 años", "Más de 5 años"],
    "calificacion": ["No calificada", "Operativa", "Técnica", "Profesional"],
}
for col, orden in ORDENES.items():
    # Control: si apareciera una categoría que no está en la lista, map() la
    # convertiría en NaN sin avisar. El assert lo frena.
    sin_mapear = set(X[col].dropna()) - set(orden)
    assert not sin_mapear, f"{col}: categorías sin orden definido {sin_mapear}"
    # Los NaN quedan como NaN: "sin dato" no tiene lugar en un orden,
    # así que lo resuelve el árbol
    X[col] = X[col].map({v: i for i, v in enumerate(orden)})

# Binarias: 0/1. En empleo_temporario los NaN quedan NaN, por la misma razón.
X["sexo"] = (X["sexo"] == "Mujer").astype(int)
X["empleo_temporario"] = X["empleo_temporario"].map({"no": 0, "si": 1})

# Nominales: no tienen orden, van en one-hot. El nulo pasa a ser una categoría
# más ("Sin dato"), así queda explícito en el árbol.
NOMINALES = ["region", "sector", "rama", "lugar_trabajo"]
X[NOMINALES] = X[NOMINALES].fillna("Sin dato")
X = pd.get_dummies(X, columns=NOMINALES, dtype=int)

print(X.shape)
# Las numéricas (tamaño, horas, ingreso) quedan con sus NaN: el árbol de
# sklearn 1.9 elige en cada corte para qué lado mandarlos
print("NaN que quedan para el árbol:")
X.isna().sum()[lambda s: s > 0]

(13938, 43)
NaN que quedan para el árbol:


calificacion                 16
tamanio_establecimiento    3869
antiguedad                 1401
empleo_temporario          2314
horas_semanales              20
ingreso_mensual            2598
dtype: int64

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEMILLA)
print("train:", X_train.shape, "  test:", X_test.shape)
pd.DataFrame({"train": y_train.value_counts(normalize=True),
              "test": y_test.value_counts(normalize=True)}).round(3)

train: (11150, 43)   test: (2788, 43)


,train,test
informal,,
0,0.624,0.624
1,0.376,0.376


In [6]:
def evaluar(modelo, X, y):
    pred = modelo.predict(X)
    return {
        "accuracy": accuracy_score(y, pred),
        "balanced_acc": balanced_accuracy_score(y, pred),
        "precision_inf": precision_score(y, pred),
        "recall_inf": recall_score(y, pred),
        "f1_inf": f1_score(y, pred),
    }

def matriz(modelo, X, y):
    cm = confusion_matrix(y, modelo.predict(X), labels=[0, 1])
    return pd.DataFrame(cm, index=["real formal", "real informal"],
                        columns=["pred formal", "pred informal"])

# Criterio entropía (lo pide la consigna) y el resto por defecto: es la línea de base
base = DecisionTreeClassifier(criterion="entropy", random_state=SEMILLA)
base.fit(X_train, y_train)

print("profundidad:", base.get_depth(), "  hojas:", base.get_n_leaves())
print(pd.DataFrame({"train": evaluar(base, X_train, y_train),
                    "test": evaluar(base, X_test, y_test)}).round(3))
# Referencia: la accuracy que tendría un modelo que dice "formal" a todos
print("Siempre 'formal' daría accuracy:", round(1 - y_test.mean(), 3))
matriz(base, X_test, y_test)

profundidad: 36   hojas: 1526
               train   test
accuracy         1.0  0.811
balanced_acc     1.0  0.794
precision_inf    1.0  0.758
recall_inf       1.0  0.729
f1_inf           1.0  0.743
Siempre 'formal' daría accuracy: 0.624


,pred formal,pred informal
real formal,1495,244
real informal,284,765


In [7]:
print(export_text(base, feature_names=list(X.columns), max_depth=3,
                  class_names=["formal", "informal"]))

|--- empleo_temporario <= 0.50
|   |--- tamanio_establecimiento <= 6.50
|   |   |--- ingreso_mensual <= 805000.00
|   |   |   |--- sector_Estatal <= 0.50
|   |   |   |   |--- truncated branch of depth 23
|   |   |   |--- sector_Estatal >  0.50
|   |   |   |   |--- truncated branch of depth 6
|   |   |--- ingreso_mensual >  805000.00
|   |   |   |--- sector_Estatal <= 0.50
|   |   |   |   |--- truncated branch of depth 16
|   |   |   |--- sector_Estatal >  0.50
|   |   |   |   |--- truncated branch of depth 6
|   |--- tamanio_establecimiento >  6.50
|   |   |--- sector_Estatal <= 0.50
|   |   |   |--- ingreso_mensual <= 645000.00
|   |   |   |   |--- truncated branch of depth 18
|   |   |   |--- ingreso_mensual >  645000.00
|   |   |   |   |--- truncated branch of depth 24
|   |   |--- sector_Estatal >  0.50
|   |   |   |--- ingreso_mensual <= 335000.00
|   |   |   |   |--- truncated branch of depth 8
|   |   |   |--- ingreso_mensual >  335000.00
|   |   |   |   |--- truncated branch of

### Evaluación del árbol base

El árbol con entropía y los parámetros por defecto llegó a profundidad 36 y 1.526
hojas. En train da 1,0 en todas las métricas y en test baja a 0,811 de accuracy y
0,743 de F1 de informal (precision 0,758, recall 0,729). Está sobreajustado: memorizó
el train.

Igual aprende algo: la accuracy de 0,811 supera el 0,624 de decir "formal" a todos.
En la matriz de confusión se equivoca para los dos lados de forma parecida: 284
informales clasificados como formales y 244 formales como informales.

Los primeros cortes son por empleo temporario, tamaño del establecimiento, ingreso
mensual y sector estatal.

## d. Combinaciones de parámetros (hold-out)

In [8]:
# Las configuraciones se definen una sola vez: el 3e usa exactamente las mismas.
# Hay algunas con un solo parámetro, para ver el efecto aislado, y otras combinadas.
CONFIGS = {
    "base":          {},
    "depth5":        {"max_depth": 5},
    "depth10":       {"max_depth": 10},
    "depth15":       {"max_depth": 15},
    "split50":       {"min_samples_split": 50},
    "split200":      {"min_samples_split": 200},
    "leaf20":        {"min_samples_leaf": 20},
    "leaf100":       {"min_samples_leaf": 100},
    "nodes20":       {"max_leaf_nodes": 20},
    "nodes100":      {"max_leaf_nodes": 100},
    "d10_leaf20":    {"max_depth": 10, "min_samples_leaf": 20},
    "d8_leaf50_n50": {"max_depth": 8, "min_samples_leaf": 50, "max_leaf_nodes": 50},
    "todos":         {"max_depth": 12, "min_samples_split": 100,
                      "min_samples_leaf": 30, "max_leaf_nodes": 150},
}

# Cada corrida se guarda en la tabla en el momento en que se hace
tabla_d = {}
for nombre, params in CONFIGS.items():
    modelo = DecisionTreeClassifier(criterion="entropy", random_state=SEMILLA, **params)
    modelo.fit(X_train, y_train)
    fila = {"params": params or "por defecto"}
    fila.update(evaluar(modelo, X_test, y_test))
    # F1 en train: si es mucho mayor que en test, hay sobreajuste
    fila["f1_inf_train"] = f1_score(y_train, modelo.predict(X_train))
    fila["profundidad"] = modelo.get_depth()
    fila["hojas"] = modelo.get_n_leaves()
    tabla_d[nombre] = fila

tabla_d = pd.DataFrame.from_dict(tabla_d, orient="index")
tabla_d.round(3)

,params,accuracy,balanced_acc,precision_inf,recall_inf,f1_inf,f1_inf_train,profundidad,hojas
base,por defecto,0.811,0.794,0.758,0.729,0.743,1.000,36,1526
depth5,{'max_depth': 5},0.839,0.833,0.772,0.811,0.791,0.802,5,31
depth10,{'max_depth': 10},0.842,0.828,0.799,0.774,0.786,0.864,10,355
depth15,{'max_depth': 15},0.826,0.810,0.784,0.744,0.763,0.944,15,992
split50,{'min_samples_split': 50},0.833,0.817,0.793,0.752,0.772,0.851,26,354
split200,{'min_samples_split': 200},0.848,0.839,0.793,0.805,0.799,0.822,17,118
leaf20,{'min_samples_leaf': 20},0.834,0.818,0.797,0.751,0.773,0.827,20,304
leaf100,{'min_samples_leaf': 100},0.843,0.835,0.783,0.805,0.794,0.809,11,72
nodes20,{'max_leaf_nodes': 20},0.840,0.830,0.787,0.789,0.788,0.800,7,20
nodes100,{'max_leaf_nodes': 100},0.848,0.838,0.797,0.799,0.798,0.835,13,100


## e. Las mismas combinaciones con validación cruzada de 5 folds

In [9]:
# Folds estratificados y fijos: todas las configuraciones ven las mismas 5 particiones
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
METRICAS = {"accuracy": "accuracy", "balanced_acc": "balanced_accuracy",
            "precision_inf": "precision", "recall_inf": "recall", "f1_inf": "f1"}

tabla_e = {}
for nombre, params in CONFIGS.items():
    modelo = DecisionTreeClassifier(criterion="entropy", random_state=SEMILLA, **params)
    cv = cross_validate(modelo, X_train, y_train, cv=folds, scoring=METRICAS)
    fila = {"params": params or "por defecto"}
    for m in METRICAS:
        fila[f"cv_{m}"] = cv[f"test_{m}"].mean()
    # Cuánto varía el F1 entre folds: es la vara para decidir si dos
    # configuraciones son distintas de verdad
    fila["cv_f1_inf_desvio"] = cv["test_f1_inf"].std()
    tabla_e[nombre] = fila

tabla_e = pd.DataFrame.from_dict(tabla_e, orient="index")
tabla_e.round(3)

,params,cv_accuracy,cv_balanced_acc,cv_precision_inf,cv_recall_inf,cv_f1_inf,cv_f1_inf_desvio
base,por defecto,0.815,0.801,0.757,0.748,0.752,0.009
depth5,{'max_depth': 5},0.842,0.831,0.790,0.789,0.790,0.008
depth10,{'max_depth': 10},0.845,0.834,0.798,0.788,0.793,0.014
depth15,{'max_depth': 15},0.828,0.816,0.775,0.765,0.770,0.011
split50,{'min_samples_split': 50},0.847,0.835,0.805,0.784,0.794,0.010
split200,{'min_samples_split': 200},0.850,0.836,0.815,0.778,0.796,0.009
leaf20,{'min_samples_leaf': 20},0.846,0.831,0.813,0.769,0.790,0.003
leaf100,{'min_samples_leaf': 100},0.843,0.830,0.802,0.777,0.789,0.010
nodes20,{'max_leaf_nodes': 20},0.839,0.829,0.788,0.785,0.786,0.009
nodes100,{'max_leaf_nodes': 100},0.852,0.840,0.809,0.794,0.801,0.012


In [10]:
# Es la comparación del punto e: ¿el hold-out y la CV ordenan igual
# las configuraciones?
comparacion = pd.DataFrame({
    "f1_holdout": tabla_d["f1_inf"],
    "f1_cv": tabla_e["cv_f1_inf"],
    "f1_cv_desvio": tabla_e["cv_f1_inf_desvio"],
    "diferencia": tabla_d["f1_inf"] - tabla_e["cv_f1_inf"],
    "hojas": tabla_d["hojas"],
})
comparacion.round(3)

,f1_holdout,f1_cv,f1_cv_desvio,diferencia,hojas
base,0.743,0.752,0.009,-0.009,1526
depth5,0.791,0.790,0.008,0.001,31
depth10,0.786,0.793,0.014,-0.006,355
depth15,0.763,0.770,0.011,-0.007,992
split50,0.772,0.794,0.010,-0.022,354
split200,0.799,0.796,0.009,0.003,118
leaf20,0.773,0.790,0.003,-0.017,304
leaf100,0.794,0.789,0.010,0.005,72
nodes20,0.788,0.786,0.009,0.002,20
nodes100,0.798,0.801,0.012,-0.003,100


## f. Conclusiones

**Nulos.** La decisión más importante del ejercicio fue no descartar filas. Los nulos
dependen del tipo de trabajo, y un `dropna()` hubiera bajado la informalidad de 37,6 %
a 27,2 % y casi borrado al servicio doméstico. El modelo hubiera parecido funcionar y
estaría midiendo otra cosa.

**Sobreajuste.** El árbol sin restricciones memoriza el train (F1 1,0 en train contra
0,743 en test). Todas las configuraciones que probé lo limitan de alguna forma, y todas
mejoran el F1 en test. Cuanto más chico el árbol, más se acercan train y test: con
profundidad 5 da 0,802 en train y 0,791 en test. Con profundidad 15 da 0,944 contra
0,763.

**Hold-out contra validación cruzada.** En la mayoría de las configuraciones el
hold-out y la CV dan parecido, con diferencias de menos de 0,01. Pero en tres el
hold-out quedó bastante por debajo de la CV: `split50` (0,772 contra 0,794), `leaf20`
(0,773 contra 0,790) y `d10_leaf20` (0,781 contra 0,794). Con un solo split,
`leaf20` parecía de las peores, y con la CV queda igual que profundidad 5. O sea que
un hold-out puede cambiar el orden de las configuraciones solo por cómo cayó el split.
La CV promedia cinco particiones y además da el desvío, que sirve para saber si dos
resultados son distintos de verdad.

**Cuál elijo.** Con la CV, la mejor es `nodes100` (F1 0,801, desvío 0,012). Muchas
configuraciones quedan a menos de un desvío de esa: `d8_leaf50_n50` (0,797),
`split200` (0,796), `depth10` (0,793), `depth5` (0,790), entre otras. Las tomo como
empatadas. Con mi regla de desempate elijo la más chica: `depth5`, con 31 hojas. Da
0,790 en CV y 0,791 en hold-out, y casi no tiene diferencia entre train y test. Un
árbol de 31 hojas se puede leer, uno de 1.526 no.

**Límites.** El mejor F1 fue 0,801: tocar los parámetros mejora
contra el árbol sin límites, pero entre las configuraciones razonables la diferencia
es chica. Además, como las filas no están ponderadas, el modelo describe la muestra de
la EPH y no la población de asalariados.